# Imports

In [2]:
pip install pytorch-ssim

  Preparing metadata (setup.py) ... done
  Created wheel for pytorch-ssim: filename=pytorch_ssim-0.1-py3-none-any.whl size=2006 sha256=78f672b38e7ed63ddb3597552f62eb2e57fb08563a4bb47e10a77f36761c3f51
  Stored in directory: /root/.cache/pip/wheels/54/a0/11/99f86224e71729ed9ef0c4ffe1b795807ad5f44bde19bc66f9
Successfully built pytorch-ssim


In [3]:
# ── Standard library ──────────────────────────────────────────────────
import os, sys, io, zipfile, math, random
from pathlib import Path
from itertools import product

# ── Numerical / visualisation ──────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ── ML analysis ───────────────────────────────────────────────────────
from sklearn.decomposition import PCA

# ── PyTorch core ──────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.ops import MLP
from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

# ── METER dependency ───────────────────────────────────────────────────
from einops import rearrange

# ── Progress / experiment tracking ────────────────────────────────────
import tqdm

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    print("wandb not installed — run: pip install wandb")

try:
    import pytorch_ssim
    SSIM_AVAILABLE = True
except ImportError:
    SSIM_AVAILABLE = False
    print("pytorch_ssim not installed — run: pip install pytorch-ssim")

print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}"
      f"  |  Devices: {torch.cuda.device_count()}")


PyTorch 2.10.0+cu128  |  CUDA: True  |  Devices: 1


# Globals

In [4]:
# ── Paths ──────────────────────────────────────────────────────────────
ROOT        = Path(".").resolve()
DATASET_DIR = ROOT / "dataset"
CKPT_DIR    = ROOT / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

NYU_DIR   = DATASET_DIR / "nyu"
KITTI_DIR = DATASET_DIR / "kitti"

# ── Compute device ─────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ── MDE image resolution (H, W) — must be divisible by 32 ─────────────
IMG_RES = {
    "nyu":   (480, 640),
    "kitti": (192, 640),
}

# ── LeJEPA self-supervised pre-training ────────────────────────────────
PRETRAIN_RES    = 256    # square crop size; lower to 128 if GPU is slow
LAMBDA          = 0.02   # SIGReg weight (λ in the paper)
N_VIEWS         = 4      # augmented views per image
PROJ_DIM        = 64     # projector output dimensionality
PRETRAIN_EPOCHS = 200
PRETRAIN_BS     = 64
PRETRAIN_LR     = 2e-3

# ── MDE supervised fine-tuning ─────────────────────────────────────────
FINETUNE_EPOCHS = 100
FINETUNE_BS     = 8
LR_BACKBONE     = 1e-4   # lower LR for the pre-trained backbone
LR_DECODER      = 1e-3   # higher LR for the freshly initialised decoder

# ── MobileViT variant — final encoder channel count ────────────────────
EMB_DIM = {"xxs": 160, "xs": 192, "s": 320}

# ── Paths to provided supervised-METER baseline weights ────────────────
METER_WEIGHTS = {
    "nyu": {
        "xxs": ROOT / "src/METER/models/build_model_best_nyu_xxs",
        "xs":  ROOT / "src/METER/models/build_model_best_nyu_xs",
        "s":   ROOT / "src/METER/models/build_model_best_nyu_s",
    },
    "kitti": {
        "xxs": ROOT / "src/METER/models/build_model_best_kitti_xxs",
        "xs":  ROOT / "src/METER/models/build_model_best_kitti_xs",
        "s":   ROOT / "src/METER/models/build_model_best_kitti_s",
    },
}


Using device: cuda


# Utils

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  LeJEPA — SIGReg (Sketched Isotropic Gaussian Regularisation)
#  Source: Balestriero & LeCun (2025), MINIMAL.md
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SIGReg(nn.Module):
    """Forces projected embeddings toward an isotropic Gaussian distribution,
    preventing representation collapse without stop-gradients or teacher nets.
    Single hyperparameter: λ (controls its weight in the total loss).
    """
    def __init__(self, knots: int = 17):
        super().__init__()
        t       = torch.linspace(0, 3, knots, dtype=torch.float32)
        dt      = 3 / (knots - 1)
        weights = torch.full((knots,), 2 * dt, dtype=torch.float32)
        weights[[0, -1]] = dt
        window  = torch.exp(-t.square() / 2.0)
        self.register_buffer("t",       t)
        self.register_buffer("phi",     window)
        self.register_buffer("weights", weights * window)

    def forward(self, proj: torch.Tensor) -> torch.Tensor:
        # proj: (V, B, proj_dim)
        A   = torch.randn(proj.size(-1), 256, device=proj.device)
        A   = A.div_(A.norm(p=2, dim=0))           # unit-norm columns
        x_t = (proj @ A).unsqueeze(-1) * self.t    # (V, B, 256, knots)
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()
        return (err @ self.weights * proj.size(-2)).mean()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  METER — Loss functions
#  Source: Papa et al. (2024), src/METER/loss.py
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _gaussian(window_size: int, sigma: float) -> torch.Tensor:
    g = torch.Tensor([math.exp(-(x - window_size // 2) ** 2 / (2 * sigma ** 2))
                      for x in range(window_size)])
    return g / g.sum()


def _create_window(window_size: int, channel: int = 1) -> torch.Tensor:
    _1d = _gaussian(window_size, 1.5).unsqueeze(1)
    _2d = _1d.mm(_1d.t()).float().unsqueeze(0).unsqueeze(0)
    return _2d.expand(channel, 1, window_size, window_size).contiguous()


def ssim_loss(img1: torch.Tensor, img2: torch.Tensor,
              val_range: float, window_size: int = 11) -> torch.Tensor:
    """Structural Similarity Index (returns the SSIM value, not 1-SSIM)."""
    _, ch, h, w = img1.size()
    real_size   = min(window_size, h, w)
    window      = _create_window(real_size, ch).to(img1.device)

    mu1     = F.conv2d(img1, window, padding=0, groups=ch)
    mu2     = F.conv2d(img2, window, padding=0, groups=ch)
    mu1_sq  = mu1.pow(2);  mu2_sq = mu2.pow(2);  mu1_mu2 = mu1 * mu2
    s1      = F.conv2d(img1 * img1, window, padding=0, groups=ch) - mu1_sq
    s2      = F.conv2d(img2 * img2, window, padding=0, groups=ch) - mu2_sq
    s12     = F.conv2d(img1 * img2, window, padding=0, groups=ch) - mu1_mu2
    C1, C2  = (0.01 * val_range) ** 2, (0.03 * val_range) ** 2
    v1, v2  = 2.0 * s12 + C2, s1 + s2 + C2
    ssim_map = ((2 * mu1_mu2 + C1) * v1) / ((mu1_sq + mu2_sq + C1) * v2)
    return ssim_map.mean()


class Sobel(nn.Module):
    def __init__(self):
        super().__init__()
        self.edge_conv = nn.Conv2d(1, 2, kernel_size=3, stride=1, padding=1, bias=False)
        kx = torch.tensor([[1., 0, -1], [2, 0, -2], [1, 0, -1]])
        ky = torch.tensor([[1., 2,  1], [0, 0,  0], [-1, -2, -1]])
        self.edge_conv.weight = nn.Parameter(
            torch.stack([kx, ky]).unsqueeze(1), requires_grad=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.edge_conv(x).view(-1, 2, x.size(2), x.size(3))


class BalancedDepthLoss(nn.Module):
    """METER supervised depth loss: L1 + gradient + surface-normal + SSIM."""
    def __init__(self):
        super().__init__()
        self.cos      = nn.CosineSimilarity(dim=1, eps=0)
        self.sobel    = Sobel()
        self.lambda_1 = 0.5
        self.lambda_2 = 100.0
        self.lambda_3 = 100.0

    def forward(self, pred: torch.Tensor, gt: torch.Tensor):
        ones = torch.ones_like(gt)

        gt_grad   = self.sobel(gt)
        pred_grad = self.sobel(pred)
        gt_dx   = gt_grad[:, 0:1];  gt_dy   = gt_grad[:, 1:2]
        pred_dx = pred_grad[:, 0:1]; pred_dy = pred_grad[:, 1:2]

        gt_normal   = torch.cat((-gt_dx,   -gt_dy,   ones), dim=1)
        pred_normal = torch.cat((-pred_dx, -pred_dy, ones), dim=1)

        loss_depth  = torch.abs(pred - gt).mean()
        loss_dx     = torch.abs(pred_dx - gt_dx).mean()
        loss_dy     = torch.abs(pred_dy - gt_dy).mean()
        loss_normal = self.lambda_2 * (1 - self.cos(pred_normal, gt_normal)).mean()
        loss_ssim   = (1 - ssim_loss(pred, gt, val_range=10.0)) * self.lambda_3
        loss_grad   = (loss_dx + loss_dy) / self.lambda_1

        total = loss_depth + loss_ssim + loss_normal + loss_grad
        return total, {"depth": loss_depth.item(), "ssim": loss_ssim.item(),
                       "normal": loss_normal.item(), "grad": loss_grad.item()}


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Depth evaluation metrics
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@torch.no_grad()
def compute_metrics(pred: torch.Tensor, gt: torch.Tensor,
                    min_depth: float = 1e-3, max_depth: float = 10.0) -> dict:
    """Compute standard MDE metrics on a single (pred, gt) batch.

    Args:
        pred, gt: (B, 1, H, W) depth tensors in metres.
    Returns:
        dict with keys: rmse, abs_rel, delta1, delta2, delta3, log_rmse
    """
    # mask out invalid / out-of-range pixels
    valid = (gt > min_depth) & (gt < max_depth)
    pred  = pred.clamp(min=min_depth, max=max_depth)

    p, g  = pred[valid], gt[valid]

    thresh = torch.maximum(p / g, g / p)
    delta1 = (thresh < 1.25   ).float().mean().item()
    delta2 = (thresh < 1.25**2).float().mean().item()
    delta3 = (thresh < 1.25**3).float().mean().item()

    abs_rel  = ((p - g).abs() / g).mean().item()
    rmse     = ((p - g).pow(2).mean()).sqrt().item()
    log_rmse = ((p.log() - g.log()).pow(2).mean()).sqrt().item()

    return dict(rmse=rmse, abs_rel=abs_rel, log_rmse=log_rmse,
                delta1=delta1, delta2=delta2, delta3=delta3)


def aggregate_metrics(metric_list: list[dict]) -> dict:
    """Average a list of per-batch metric dicts."""
    keys = metric_list[0].keys()
    return {k: float(np.mean([m[k] for m in metric_list])) for k in keys}


def print_metrics(tag: str, metrics: dict):
    print(f"\n{'─'*55}")
    print(f"  {tag}")
    print(f"{'─'*55}")
    print(f"  RMSE      : {metrics['rmse']:.4f} m")
    print(f"  AbsRel    : {metrics['abs_rel']:.4f}")
    print(f"  log-RMSE  : {metrics['log_rmse']:.4f}")
    print(f"  δ₁ (<1.25): {metrics['delta1']*100:.2f} %")
    print(f"  δ₂        : {metrics['delta2']*100:.2f} %")
    print(f"  δ₃        : {metrics['delta3']*100:.2f} %")
    print(f"{'─'*55}\n")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  PCA probing helper — zero-shot geometric analysis
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@torch.no_grad()
def extract_features(model: nn.Module, x: torch.Tensor):
    """Run the MobileViT encoder and return (final_feat, skip_list).

    Args:
        model: a build_METER_model instance (encoder + decoder).
        x: (B, 3, H, W) input batch.
    Returns:
        feat  : (B, C, h, w)  final feature map before decoder
        skips : list[(B, Ci, Hi, Wi)]  skip-connection feature maps
    """
    model.eval()
    feat, skips = model.encoder(x.to(DEVICE))
    return feat, skips


def pca_feature_map(feat: torch.Tensor, n_components: int = 3) -> np.ndarray:
    """Project a spatial feature map to n_components via PCA.

    Args:
        feat: (B, C, H, W) tensor.
    Returns:
        pca_rgb: (B, H, W, n_components) array, each channel normalised to [0,1].
    """
    B, C, H, W = feat.shape
    X = feat.permute(0, 2, 3, 1).reshape(B * H * W, C).cpu().float().numpy()
    pca  = PCA(n_components=n_components)
    proj = pca.fit_transform(X).reshape(B, H, W, n_components)
    # normalise each component to [0, 1] for display
    for c in range(n_components):
        mn, mx = proj[..., c].min(), proj[..., c].max()
        proj[..., c] = (proj[..., c] - mn) / (mx - mn + 1e-8)
    return proj


def denorm(t: torch.Tensor) -> np.ndarray:
    """Undo ImageNet normalisation for display."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


def visualise_pca_probing(model: nn.Module, loader: DataLoader,
                           n_images: int = 4, skip_idx: int = 3):
    """Show RGB | GT depth | PCA of skip feature y{skip_idx} side-by-side.

    skip_idx: 0=y0 (H/2), 1=y1 (H/4), 2=y2 (H/8), 3=y3 (H/16)
    """
    model.eval()
    rgb_batch, depth_batch = next(iter(loader))
    rgb_batch  = rgb_batch[:n_images].to(DEVICE)
    depth_batch = depth_batch[:n_images]

    feat, skips = extract_features(model, rgb_batch)
    pca_maps    = pca_feature_map(skips[skip_idx])   # (B, H, W, 3)

    fig, axes = plt.subplots(n_images, 3, figsize=(12, 4 * n_images))
    for i in range(n_images):
        axes[i, 0].imshow(denorm(rgb_batch[i]))
        axes[i, 0].set_title("RGB input")
        axes[i, 1].imshow(depth_batch[i, 0].cpu(), cmap="plasma")
        axes[i, 1].set_title("GT depth")
        axes[i, 2].imshow(pca_maps[i])
        axes[i, 2].set_title(f"PCA(y{skip_idx}) — 3 components")
        for ax in axes[i]:
            ax.axis("off")
    plt.tight_layout()
    plt.show()
    return fig


# Data

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Dataset download instructions
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Run these shell commands ONCE to populate dataset/:
#
#   NYU Depth V2 (~2.8 GB):
#     mkdir -p dataset/nyu
#     wget https://s3-eu-west-1.amazonaws.com/densedepth/nyu_data.zip \
#          -O dataset/nyu/nyu_data.zip
#   (leave the zip in place — the loader reads directly from it)
#
#   KITTI Eigen split (~10 GB):
#     mkdir -p dataset/kitti
#     wget https://s3-eu-west-1.amazonaws.com/densedepth/KITTI.zip \
#          -O dataset/kitti/KITTI.zip
#   (same — keep the zip, loader reads from it)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  NYU Depth V2 Dataset
#  Format: zip → data/nyu2_{train,test}.csv with paired JPEG+16-bit PNG
#  Depth unit: uint16 PNG, value / 1000 = metres
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class NYUDataset(Dataset):
    """NYU Depth V2 (DenseDepth preprocessed split).

    Args:
        root       : path containing ``nyu_data.zip``
        split      : ``'train'`` or ``'test'``
        return_depth: if True, returns (rgb, depth); if False, returns V
                      augmented views (for LeJEPA SSL pre-training)
        n_views    : number of SSL views (used only when return_depth=False)
        img_size   : (H, W) target size for supervised fine-tuning mode
        pretrain_res: square crop size for SSL mode
    """
    NYU_MAX_DEPTH = 10.0   # metres

    def __init__(self, root: str | Path, split: str = "train",
                 return_depth: bool = True, n_views: int = N_VIEWS,
                 img_size: tuple = IMG_RES["nyu"],
                 pretrain_res: int = PRETRAIN_RES):
        self.zip_path    = Path(root) / "nyu_data.zip"
        assert self.zip_path.exists(), \
            f"NYU zip not found: {self.zip_path}\n" \
            "  → wget https://s3-eu-west-1.amazonaws.com/densedepth/nyu_data.zip"
        self.return_depth = return_depth
        self.n_views      = n_views
        self.img_size     = img_size   # (H, W)

        archive  = zipfile.ZipFile(str(self.zip_path), "r")
        csv_key  = "data/nyu2_train.csv" if split == "train" else "data/nyu2_test.csv"
        csv_data = archive.read(csv_key).decode("utf-8")
        self.pairs = [row.split(",") for row in csv_data.strip().split("\n") if row]
        archive.close()

        # ── Supervised fine-tuning transforms ────────────────────────────
        self.ft_rgb = v2.Compose([
            v2.Resize(img_size),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        # ── SSL pre-training augmentation (multi-view, no depth) ─────────
        self.ssl_aug = v2.Compose([
            v2.RandomResizedCrop(pretrain_res, scale=(0.08, 1.0)),
            v2.RandomApply([v2.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
            v2.RandomGrayscale(p=0.2),
            v2.RandomApply([v2.GaussianBlur(kernel_size=7, sigma=(0.1, 2.0))]),
            v2.RandomApply([v2.RandomSolarize(threshold=128)], p=0.2),
            v2.RandomHorizontalFlip(),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    # internal helpers — reopen zip per worker to be fork-safe
    def _open(self):
        return zipfile.ZipFile(str(self.zip_path), "r")

    def _read_rgb(self, archive, path: str) -> Image.Image:
        return Image.open(io.BytesIO(archive.read(path.strip()))).convert("RGB")

    def _read_depth(self, archive, path: str) -> torch.Tensor:
        depth_np = np.array(
            Image.open(io.BytesIO(archive.read(path.strip()))), dtype=np.float32
        ) / 1000.0   # uint16 mm → float metres
        return torch.from_numpy(depth_np).unsqueeze(0)   # (1, H, W)

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int):
        rgb_path, depth_path = self.pairs[idx][0], self.pairs[idx][1]
        archive = self._open()
        try:
            img = self._read_rgb(archive, rgb_path)
            if not self.return_depth:
                # SSL mode — return V views, no depth label
                views = torch.stack([self.ssl_aug(img) for _ in range(self.n_views)])
                return views                             # (V, 3, H, W)
            else:
                depth = self._read_depth(archive, depth_path)
                rgb_t = self.ft_rgb(img)                # (3, H, W)
                depth_t = F.interpolate(
                    depth.unsqueeze(0), size=self.img_size, mode="nearest"
                ).squeeze(0).clamp(1e-3, self.NYU_MAX_DEPTH)  # (1, H, W)
                return rgb_t, depth_t
        finally:
            archive.close()


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  KITTI Eigen-Split Dataset
#  Format: zip → data/kitti_train.csv / kitti_test.csv
#  Depth unit: uint16 PNG, value / 256 = metres (velodyne projection)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class KITTIDataset(Dataset):
    """KITTI Eigen split (DenseDepth preprocessed).

    Same API as NYUDataset: return_depth=False for LeJEPA SSL pre-training,
    return_depth=True for supervised fine-tuning.
    """
    KITTI_MAX_DEPTH = 80.0   # metres

    def __init__(self, root: str | Path, split: str = "train",
                 return_depth: bool = True, n_views: int = N_VIEWS,
                 img_size: tuple = IMG_RES["kitti"],
                 pretrain_res: int = PRETRAIN_RES):
        self.zip_path     = Path(root) / "KITTI.zip"
        assert self.zip_path.exists(), \
            f"KITTI zip not found: {self.zip_path}\n" \
            "  → wget https://s3-eu-west-1.amazonaws.com/densedepth/KITTI.zip"
        self.return_depth = return_depth
        self.n_views      = n_views
        self.img_size     = img_size

        archive  = zipfile.ZipFile(str(self.zip_path), "r")
        csv_key  = "data/kitti_train.csv" if split == "train" else "data/kitti_test.csv"
        csv_data = archive.read(csv_key).decode("utf-8")
        self.pairs = [row.split(",") for row in csv_data.strip().split("\n") if row]
        archive.close()

        self.ft_rgb = v2.Compose([
            v2.Resize(img_size),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        self.ssl_aug = v2.Compose([
            v2.RandomResizedCrop(pretrain_res, scale=(0.08, 1.0)),
            v2.RandomApply([v2.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
            v2.RandomGrayscale(p=0.2),
            v2.RandomApply([v2.GaussianBlur(kernel_size=7, sigma=(0.1, 2.0))]),
            v2.RandomApply([v2.RandomSolarize(threshold=128)], p=0.2),
            v2.RandomHorizontalFlip(),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def _open(self):
        return zipfile.ZipFile(str(self.zip_path), "r")

    def _read_rgb(self, archive, path: str) -> Image.Image:
        return Image.open(io.BytesIO(archive.read(path.strip()))).convert("RGB")

    def _read_depth(self, archive, path: str) -> torch.Tensor:
        depth_np = np.array(
            Image.open(io.BytesIO(archive.read(path.strip()))), dtype=np.float32
        ) / 256.0   # uint16 → float metres (KITTI convention)
        return torch.from_numpy(depth_np).unsqueeze(0)   # (1, H, W)

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int):
        rgb_path, depth_path = self.pairs[idx][0], self.pairs[idx][1]
        archive = self._open()
        try:
            img = self._read_rgb(archive, rgb_path)
            if not self.return_depth:
                views = torch.stack([self.ssl_aug(img) for _ in range(self.n_views)])
                return views                              # (V, 3, H, W)
            else:
                depth   = self._read_depth(archive, depth_path)
                rgb_t   = self.ft_rgb(img)
                depth_t = F.interpolate(
                    depth.unsqueeze(0), size=self.img_size, mode="nearest"
                ).squeeze(0).clamp(1e-3, self.KITTI_MAX_DEPTH)
                return rgb_t, depth_t
        finally:
            archive.close()


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DataLoader factory
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

_DATASET_CLS = {"nyu": NYUDataset, "kitti": KITTIDataset}
_DATASET_DIR = {"nyu": NYU_DIR,    "kitti": KITTI_DIR}

def get_loader(dataset: str, split: str, phase: str,
               batch_size: int | None = None,
               num_workers: int = 4) -> DataLoader:
    """Build a DataLoader for the requested dataset / phase.

    Args:
        dataset   : ``'nyu'`` or ``'kitti'``
        split     : ``'train'`` or ``'test'``
        phase     : ``'pretrain'``  → SSL mode (no depth, V views)
                    ``'finetune'``  → supervised mode (rgb + depth)
                    ``'eval'``      → supervised, no augmentation
        batch_size: overrides the global default for the phase
        num_workers: DataLoader workers (set 0 when debugging)
    """
    return_depth = (phase in ("finetune", "eval"))
    bs = batch_size or (PRETRAIN_BS if phase == "pretrain" else FINETUNE_BS)

    ds = _DATASET_CLS[dataset](
        root=_DATASET_DIR[dataset],
        split=split,
        return_depth=return_depth,
        n_views=N_VIEWS,
    )
    shuffle = (split == "train")
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      num_workers=num_workers, pin_memory=True,
                      drop_last=(phase == "pretrain"))


# ── Quick sanity-check (run after downloading the datasets) ───────────
def check_datasets():
    for ds_name in ("nyu", "kitti"):
        zip_path = _DATASET_DIR[ds_name] / ("nyu_data.zip" if ds_name == "nyu" else "KITTI.zip")
        status   = "✓  found" if zip_path.exists() else "✗  MISSING"
        print(f"  {ds_name.upper():<6} {status}  ({zip_path})")

print("Dataset paths:")
check_datasets()


# Network

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  METER — Encoder (MobileViT backbone)
#  Source: Papa et al. (2024), src/METER/architecture.py
#  Fixes applied vs original:
#    • removed `from globals import RGB_img_res` (now uses IMG_RES dict)
#    • removed `device='cuda:0'` from SeparableConv2d (move .to(DEVICE) to
#      model-level, standard PyTorch practice)
#    • fixed syntax error in mobilevit_xs() (mismatched parenthesis)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, depth=1, bias=False):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, out_channels * depth,
                                   kernel_size=kernel_size, groups=depth,
                                   padding=1, stride=stride, bias=bias)
        self.pointwise = nn.Conv2d(out_channels * depth, out_channels,
                                   kernel_size=1, bias=bias)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


def conv_1x1_bn(inp, oup):
    return nn.Sequential(nn.Conv2d(inp, oup, 1, 1, 0, bias=False),
                         nn.BatchNorm2d(oup), nn.ReLU())


def conv_nxn_bn(inp, oup, kernel_size=3, stride=1):
    return nn.Sequential(
        SeparableConv2d(inp, oup, kernel_size, stride=stride, bias=False),
        nn.BatchNorm2d(oup), nn.ReLU())


class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn   = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class Attention(nn.Module):
    def __init__(self, dim, heads=8, dim_head=64, dropout=0.):
        super().__init__()
        inner_dim    = dim_head * heads
        project_out  = not (heads == 1 and dim_head == dim)
        self.heads   = heads
        self.scale   = dim_head ** -0.5
        self.attend  = nn.Softmax(dim=-1)
        self.to_qkv  = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out  = (nn.Sequential(nn.Linear(inner_dim, dim),
                                      nn.Dropout(dropout))
                        if project_out else nn.Identity())

    def forward(self, x):
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, "b p n (h d) -> b p h n d",
                                          h=self.heads), qkv)
        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        out  = torch.matmul(self.attend(dots), v)
        out  = rearrange(out, "b p h n d -> b p n (h d)")
        return self.to_out(out)


class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0.):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleList([
                PreNorm(dim, Attention(dim, heads, dim_head, dropout)),
                PreNorm(dim, FeedForward(dim, mlp_dim, dropout)),
            ])
            for _ in range(depth)
        ])

    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x)   + x
        return x


class MV2Block(nn.Module):
    def __init__(self, inp, oup, stride=1, expansion=4):
        super().__init__()
        self.stride = stride
        hidden_dim  = int(inp * expansion)
        self.use_res_connect = (stride == 1 and inp == oup)
        if expansion == 1:
            self.conv = nn.Sequential(
                nn.Conv2d(hidden_dim, hidden_dim, 3, stride, 1,
                          groups=hidden_dim, bias=False),
                nn.BatchNorm2d(hidden_dim), nn.ReLU(),
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup))
        else:
            self.conv = nn.Sequential(
                nn.Conv2d(inp, hidden_dim, 1, 1, 0, bias=False),
                nn.BatchNorm2d(hidden_dim), nn.ReLU(),
                nn.Conv2d(hidden_dim, hidden_dim, 3, stride, 1,
                          groups=hidden_dim, bias=False),
                nn.BatchNorm2d(hidden_dim), nn.ReLU(),
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup))

    def forward(self, x):
        return x + self.conv(x) if self.use_res_connect else self.conv(x)


class MobileViTBlock(nn.Module):
    def __init__(self, dim, depth, channel, kernel_size, patch_size, mlp_dim,
                 dropout=0.):
        super().__init__()
        self.ph, self.pw = patch_size
        self.conv1       = conv_nxn_bn(channel, channel, kernel_size)
        self.conv2       = conv_1x1_bn(channel, dim)
        self.transformer = Transformer(dim, depth, 4, 8, mlp_dim, dropout)
        self.conv3       = conv_1x1_bn(dim, channel)
        self.conv4       = conv_nxn_bn(2 * channel, channel, kernel_size)

    def forward(self, x):
        y = x.clone()
        x = self.conv2(self.conv1(x))
        _, _, h, w = x.shape
        x = rearrange(x, "b d (h ph) (w pw) -> b (ph pw) (h w) d",
                      ph=self.ph, pw=self.pw)
        x = self.transformer(x)
        x = rearrange(x, "b (ph pw) (h w) d -> b d (h ph) (w pw)",
                      h=h // self.ph, w=w // self.pw, ph=self.ph, pw=self.pw)
        x = self.conv3(x)
        x = self.conv4(torch.cat((x, y), dim=1))
        return x


class MobileViT(nn.Module):
    """Encoder backbone shared by all three METER variants (xxs / xs / s)."""
    def __init__(self, image_size, dims, channels, expansion=4,
                 kernel_size=3, patch_size=(2, 2)):
        super().__init__()
        ph, pw = patch_size
        assert image_size[0] % ph == 0 and image_size[1] % pw == 0
        L = [1, 1, 1]

        self.conv1 = conv_nxn_bn(3, channels[0], stride=2)
        self.mv2   = nn.ModuleList([
            MV2Block(channels[0], channels[1], 1, expansion),
            MV2Block(channels[1], channels[2], 2, expansion),
            MV2Block(channels[2], channels[3], 1, expansion),
            MV2Block(channels[2], channels[3], 1, expansion),  # repeat
            MV2Block(channels[3], channels[4], 2, expansion),
            MV2Block(channels[5], channels[6], 2, expansion),
            MV2Block(channels[7], channels[8], 2, expansion),
        ])
        self.mvit  = nn.ModuleList([
            MobileViTBlock(dims[0], L[0], channels[5], kernel_size,
                           patch_size, int(dims[0] * 2)),
            MobileViTBlock(dims[1], L[1], channels[7], kernel_size,
                           patch_size, int(dims[1] * 4)),
            MobileViTBlock(dims[2], L[2], channels[9], kernel_size,
                           patch_size, int(dims[2] * 4)),
        ])
        self.conv2 = conv_1x1_bn(channels[-2], channels[-1])

    def forward(self, x):
        y0 = self.conv1(x)
        x  = self.mv2[0](y0)

        y1 = self.mv2[1](x)
        x  = self.mv2[3](self.mv2[2](y1))

        y2 = self.mv2[4](x)
        x  = self.mvit[0](y2)

        y3 = self.mv2[5](x)
        x  = self.mvit[1](y3)

        x  = self.mv2[6](x)
        x  = self.mvit[2](x)
        x  = self.conv2(x)

        return x, [y0, y1, y2, y3]   # feat + 4 skip-connection maps


# ── Variant factory functions ─────────────────────────────────────────

def mobilevit_xxs(image_size):
    dims     = [64, 80, 96]
    channels = [16, 16, 24, 24, 48, 48, 64, 64, 80, 80, 160]
    return MobileViT(image_size, dims, channels, expansion=2), "xxs"


def mobilevit_xs(image_size):
    dims     = [96, 120, 144]
    channels = [16, 32, 48, 48, 64, 64, 80, 80, 96, 96, 192]
    return MobileViT(image_size, dims, channels), "xs"


def mobilevit_s(image_size):
    dims     = [144, 192, 240]
    channels = [16, 32, 64, 64, 96, 96, 128, 128, 160, 160, 320]
    return MobileViT(image_size, dims, channels), "s"


_BACKBONE_FN = {"xxs": mobilevit_xxs, "xs": mobilevit_xs, "s": mobilevit_s}


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  METER — Decoder + full model
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class UpSampleLayer(nn.Module):
    """Transpose-conv upsample + skip-connection concat + SepConv."""
    def __init__(self, inp, oup, sep_conv_filters):
        super().__init__()
        self.up  = nn.ConvTranspose2d(inp, oup, kernel_size=3, stride=2,
                                      padding=1, output_padding=1, bias=False)
        self.end = nn.Sequential(
            SeparableConv2d(sep_conv_filters, oup, kernel_size=3),
            nn.ReLU())

    def forward(self, x, skip):
        x = self.up(x)
        # handle off-by-one from odd spatial dims
        if x.shape[-1] != skip.shape[-1]:
            skip = F.pad(skip, (0, x.shape[-1] - skip.shape[-1]))
        if x.shape[-2] != skip.shape[-2]:
            skip = F.pad(skip, (0, 0, 0, x.shape[-2] - skip.shape[-2]))
        x = torch.cat([x, skip], dim=1)
        return self.end(x)


class METERDecoder(nn.Module):
    """U-Net-style depth decoder for all three MobileViT variants."""
    # (inp_ch, skip_ch) for each up-block, indexed by variant
    _CFG = {
        "s":   [(320, 128, 128, 192), (64, 32, 128, 96), (32, 16, 64, 80)],
        "xs":  [(192, 128, 128, 144), (64, 32, 128, 96), (32, 16, 64, 64)],
        "xxs": [(160,  64,  64,  96), (32, 16,  64, 64), (16,  8, 32, 32)],
    }

    def __init__(self, variant: str):
        super().__init__()
        cfg = self._CFG[variant]
        # conv_in: reduce final encoder channels to first decoder width
        self.conv_in = nn.Conv2d(cfg[0][0], cfg[0][1], kernel_size=1,
                                 padding=0, bias=False)
        # three up-sample blocks;  sep_conv_filters = up_out + skip_ch
        self.up1 = UpSampleLayer(cfg[0][1], cfg[1][0], cfg[0][3])
        self.up2 = UpSampleLayer(cfg[1][0], cfg[2][0], cfg[1][3])
        self.up3 = UpSampleLayer(cfg[2][0], cfg[2][1], cfg[2][3])
        self.conv_out = nn.Conv2d(cfg[2][1], 1, kernel_size=3, padding=1,
                                  bias=False)

    def forward(self, x, skips):
        # skips = [y0, y1, y2, y3] from MobileViT.forward()
        x = self.conv_in(x)
        x = self.up1(x, skips[3])
        x = self.up2(x, skips[2])
        x = self.up3(x, skips[1])
        return self.conv_out(x)            # (B, 1, H/2, W/2) approx


class METERModel(nn.Module):
    """Full METER model: MobileViT encoder + depth decoder."""
    def __init__(self, variant: str = "xxs",
                 dataset: str = "nyu"):
        super().__init__()
        img_size      = IMG_RES[dataset]
        self.encoder, _ = _BACKBONE_FN[variant](img_size)
        self.decoder    = METERDecoder(variant)
        self.variant    = variant

    def forward(self, x):
        feat, skips = self.encoder(x)
        depth       = self.decoder(feat, skips)
        # upsample prediction to match input resolution
        depth = F.interpolate(depth, size=x.shape[-2:],
                              mode="bilinear", align_corners=False)
        return F.relu(depth)               # depth must be non-negative


def load_meter_baseline(variant: str, dataset: str) -> METERModel:
    """Load the provided supervised-METER checkpoint."""
    model = METERModel(variant=variant, dataset=dataset).to(DEVICE)
    ckpt  = METER_WEIGHTS[dataset][variant]
    assert Path(ckpt).exists(), f"Checkpoint not found: {ckpt}"
    state = torch.load(ckpt, map_location=DEVICE)
    # the provided checkpoints may be bare state-dicts or wrapped
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    model.load_state_dict(state, strict=False)
    model.eval()
    print(f"Loaded supervised baseline: METER-{variant.upper()} / {dataset.upper()}")
    return model


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  LeJEPA wrapper — adds GAP + projector on top of MobileViT for SSL
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class MobileViTLeJEPA(nn.Module):
    """MobileViT backbone adapted for LeJEPA self-supervised pre-training.

    Architecture:
        MobileViT encoder  →  GlobalAvgPool  →  Projector MLP
        ↓                                        ↓
        emb (B*V, emb_dim)                   proj (V, B, proj_dim)

    After pre-training, only the backbone weights are saved and re-loaded
    into a standard METERModel for supervised fine-tuning.  The projector
    and pooling head are discarded.
    """
    def __init__(self, variant: str = "xxs",
                 dataset: str = "nyu",
                 proj_dim: int = PROJ_DIM):
        super().__init__()
        img_size       = IMG_RES[dataset]
        emb_dim        = EMB_DIM[variant]
        self.backbone, _ = _BACKBONE_FN[variant](img_size)
        self.pool      = nn.AdaptiveAvgPool2d(1)
        self.proj      = MLP(emb_dim,
                             hidden_channels=[2048, 2048, proj_dim],
                             norm_layer=nn.BatchNorm1d)
        self.emb_dim   = emb_dim
        self.proj_dim  = proj_dim

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: (B, V, 3, H, W)  — V augmented views per image
        Returns:
            emb : (B*V, emb_dim)   — backbone embeddings (for probing)
            proj: (V,   B, proj_dim) — projected embeddings (for LeJEPA loss)
        """
        B, V = x.shape[:2]
        flat = x.flatten(0, 1)                  # (B*V, 3, H, W)
        feat, _  = self.backbone(flat)           # (B*V, C, h, w)
        emb  = self.pool(feat).flatten(1)        # (B*V, emb_dim)
        proj = self.proj(emb).reshape(B, V, -1)  # (B, V, proj_dim)
        proj = proj.transpose(0, 1)              # (V, B, proj_dim)
        return emb, proj


# Train

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Phase 2 — LeJEPA Self-Supervised Pre-training
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def pretrain_lejepa(variant: str = "xxs",
                    dataset: str = "nyu",
                    epochs:  int = PRETRAIN_EPOCHS,
                    use_wandb: bool = False) -> MobileViTLeJEPA:
    """LeJEPA SSL pre-training of the MobileViT backbone.

    Saves backbone weights to:
        checkpoints/lejepa_{variant}_{dataset}.pth

    Returns the trained MobileViTLeJEPA model.
    """
    run_name = f"lejepa_{variant}_{dataset}"
    print(f"\n{'='*60}")
    print(f"  LeJEPA pre-training: {variant.upper()} on {dataset.upper()}")
    print(f"  Epochs: {epochs} | BS: {PRETRAIN_BS} | λ: {LAMBDA} | V: {N_VIEWS}")
    print(f"{'='*60}\n")

    if use_wandb and WANDB_AVAILABLE:
        wandb.init(project="lejepa-meter", name=run_name,
                   config=dict(variant=variant, dataset=dataset, epochs=epochs,
                               lam=LAMBDA, n_views=N_VIEWS, proj_dim=PROJ_DIM,
                               bs=PRETRAIN_BS, lr=PRETRAIN_LR))

    # ── Model, loss, optimiser ────────────────────────────────────────
    net    = MobileViTLeJEPA(variant=variant, dataset=dataset).to(DEVICE)
    sigreg = SIGReg().to(DEVICE)

    opt = torch.optim.AdamW(net.parameters(), lr=PRETRAIN_LR,
                            weight_decay=5e-2)
    loader      = get_loader(dataset, "train", "pretrain")
    total_steps = len(loader) * epochs
    warmup      = len(loader)          # 1 epoch warm-up
    sched = SequentialLR(opt, schedulers=[
        LinearLR(opt, start_factor=0.01, total_iters=warmup),
        CosineAnnealingLR(opt, T_max=total_steps - warmup, eta_min=1e-5),
    ], milestones=[warmup])

    scaler = GradScaler(enabled=(DEVICE == "cuda"))

    history = {"lejepa": [], "sigreg": [], "inv": []}

    for epoch in range(1, epochs + 1):
        net.train()
        ep_lejepa = ep_sig = ep_inv = 0.0

        for views in tqdm.tqdm(loader, desc=f"[{run_name}] Epoch {epoch}/{epochs}",
                                leave=False):
            views = views.to(DEVICE, non_blocking=True)   # (B, V, 3, H, W)

            with autocast(DEVICE, dtype=torch.bfloat16):
                _, proj     = net(views)
                inv_loss    = (proj.mean(0) - proj).square().mean()
                sigreg_loss = sigreg(proj)
                loss        = sigreg_loss * LAMBDA + inv_loss * (1 - LAMBDA)

            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()

            ep_lejepa += loss.item()
            ep_sig    += sigreg_loss.item()
            ep_inv    += inv_loss.item()

        n = len(loader)
        ep_lejepa /= n;  ep_sig /= n;  ep_inv /= n
        history["lejepa"].append(ep_lejepa)
        history["sigreg"].append(ep_sig)
        history["inv"].append(ep_inv)

        print(f"  Epoch {epoch:>3} | lejepa={ep_lejepa:.4f} "
              f"sigreg={ep_sig:.4f}  inv={ep_inv:.4f}")

        if use_wandb and WANDB_AVAILABLE:
            wandb.log({"epoch": epoch, "train/lejepa": ep_lejepa,
                       "train/sigreg": ep_sig, "train/inv": ep_inv})

    # ── Save backbone-only weights ────────────────────────────────────
    ckpt_path = CKPT_DIR / f"lejepa_{variant}_{dataset}.pth"
    torch.save(net.backbone.state_dict(), ckpt_path)
    print(f"\n✓  Backbone saved → {ckpt_path}")

    if use_wandb and WANDB_AVAILABLE:
        wandb.finish()

    return net, history


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Phase 4 — Supervised fine-tuning for depth estimation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def finetune_meter(variant: str = "xxs",
                   dataset: str = "nyu",
                   pretrain_ckpt: str | Path | None = None,
                   epochs: int = FINETUNE_EPOCHS,
                   use_wandb: bool = False) -> tuple[METERModel, dict]:
    """Supervised depth fine-tuning of METER.

    Args:
        pretrain_ckpt: path to a LeJEPA backbone checkpoint (.pth).
                       Pass ``None`` to train from random initialisation
                       (the ablation baseline).
    Saves model to:
        checkpoints/meter_{variant}_{dataset}_{tag}.pth
    """
    tag      = "lejepa"   if pretrain_ckpt else "random"
    run_name = f"meter_{variant}_{dataset}_{tag}"
    print(f"\n{'='*60}")
    print(f"  Fine-tuning METER: {variant.upper()} on {dataset.upper()} "
          f"(pre-train: {tag})")
    print(f"  Epochs: {epochs} | BS: {FINETUNE_BS}")
    print(f"{'='*60}\n")

    if use_wandb and WANDB_AVAILABLE:
        wandb.init(project="lejepa-meter", name=run_name,
                   config=dict(variant=variant, dataset=dataset,
                               pretrain=tag, epochs=epochs))

    # ── Build model ───────────────────────────────────────────────────
    model = METERModel(variant=variant, dataset=dataset).to(DEVICE)

    if pretrain_ckpt is not None:
        ckpt = Path(pretrain_ckpt)
        assert ckpt.exists(), f"Pre-train checkpoint not found: {ckpt}"
        model.encoder.load_state_dict(
            torch.load(ckpt, map_location=DEVICE), strict=True)
        print(f"  Loaded LeJEPA backbone from {ckpt.name}")
    else:
        print("  Random initialisation (ablation baseline)")

    # ── Differential learning rates ───────────────────────────────────
    opt = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": LR_BACKBONE,
         "weight_decay": 1e-2},
        {"params": model.decoder.parameters(), "lr": LR_DECODER,
         "weight_decay": 1e-4},
    ])
    criterion = BalancedDepthLoss().to(DEVICE)
    train_loader = get_loader(dataset, "train", "finetune")
    total_steps  = len(train_loader) * epochs
    warmup       = len(train_loader)
    sched = SequentialLR(opt, schedulers=[
        LinearLR(opt, start_factor=0.1, total_iters=warmup),
        CosineAnnealingLR(opt, T_max=total_steps - warmup, eta_min=1e-6),
    ], milestones=[warmup])

    scaler = GradScaler(enabled=(DEVICE == "cuda"))
    history = {"loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss = 0.0

        for rgb, depth in tqdm.tqdm(train_loader,
                                     desc=f"[{run_name}] Epoch {epoch}/{epochs}",
                                     leave=False):
            rgb   = rgb.to(DEVICE,   non_blocking=True)
            depth = depth.to(DEVICE, non_blocking=True)

            with autocast(DEVICE, dtype=torch.bfloat16):
                pred        = model(rgb)
                total, _    = criterion(pred, depth)

            opt.zero_grad()
            scaler.scale(total).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            ep_loss += total.item()

        ep_loss /= len(train_loader)
        history["loss"].append(ep_loss)
        print(f"  Epoch {epoch:>3} | loss={ep_loss:.4f}")

        if use_wandb and WANDB_AVAILABLE:
            wandb.log({"epoch": epoch, "train/loss": ep_loss})

    ckpt_path = CKPT_DIR / f"{run_name}.pth"
    torch.save(model.state_dict(), ckpt_path)
    print(f"\n✓  Model saved → {ckpt_path}")

    if use_wandb and WANDB_AVAILABLE:
        wandb.finish()

    return model, history


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Execution — run all training experiments
#
#  Recommended order (parallelise across two GPUs where possible):
#   GPU-A (Colab)    : LeJEPA pre-training  (longer, no depth needed)
#   GPU-B (RTX 5060) : Fine-tuning + ablations
#
#  Set USE_WANDB = True if you have wandb configured.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

USE_WANDB = False   # set True to log to Weights & Biases

VARIANTS  = ["xxs", "xs", "s"]   # start with xxs, enable xs/s when ready
DATASETS  = ["nyu", "kitti"]

# ── Step 1: LeJEPA pre-training ───────────────────────────────────────
# For each variant × dataset combination, pre-train the backbone with
# the LeJEPA objective and save the backbone weights.
# Expected time: ~2h (xxs) / ~4h (xs) / ~8h (s) per dataset.

for variant in VARIANTS[:1]:          # start with xxs only
    for ds in DATASETS:
        pretrain_lejepa(variant=variant, dataset=ds,
                        epochs=PRETRAIN_EPOCHS, use_wandb=USE_WANDB)

# ── Step 2: Fine-tuning — LeJEPA pre-trained backbone ─────────────────
for variant in VARIANTS[:1]:
    for ds in DATASETS:
        ckpt = CKPT_DIR / f"lejepa_{variant}_{ds}.pth"
        finetune_meter(variant=variant, dataset=ds,
                       pretrain_ckpt=ckpt,
                       epochs=FINETUNE_EPOCHS, use_wandb=USE_WANDB)

# ── Step 3: Fine-tuning — random-init ablation baseline ───────────────
for variant in VARIANTS[:1]:
    for ds in DATASETS:
        finetune_meter(variant=variant, dataset=ds,
                       pretrain_ckpt=None,          # random init
                       epochs=FINETUNE_EPOCHS, use_wandb=USE_WANDB)


# Test

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Phase 1 — Evaluate supervised METER baseline
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@torch.no_grad()
def evaluate(model: nn.Module, dataset: str,
             split: str = "test") -> dict:
    """Run full evaluation of a METERModel on a test split.

    Returns a dict of averaged metrics (rmse, abs_rel, delta1, …).
    """
    model.eval()
    loader = get_loader(dataset, split, "eval", batch_size=4)
    all_metrics = []
    max_d = NYUDataset.NYU_MAX_DEPTH if dataset == "nyu" else KITTIDataset.KITTI_MAX_DEPTH

    for rgb, depth in tqdm.tqdm(loader, desc=f"Eval {dataset}", leave=False):
        rgb   = rgb.to(DEVICE)
        depth = depth.to(DEVICE)
        pred  = model(rgb)
        pred  = F.interpolate(pred, size=depth.shape[-2:],
                              mode="bilinear", align_corners=False)
        all_metrics.append(compute_metrics(pred, depth, max_depth=max_d))

    return aggregate_metrics(all_metrics)


# ── Run: supervised-baseline evaluation ──────────────────────────────
baseline_results = {}
for variant in VARIANTS[:1]:
    for ds in DATASETS:
        key   = f"supervised_{variant}_{ds}"
        model = load_meter_baseline(variant, ds)
        m     = evaluate(model, ds)
        baseline_results[key] = m
        print_metrics(f"Supervised METER-{variant.upper()} / {ds.upper()}", m)


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Phase 3 — Zero-shot PCA probing
#  (Run AFTER LeJEPA pre-training, BEFORE fine-tuning)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def run_pca_probing(variant: str = "xxs", dataset: str = "nyu"):
    """Load a LeJEPA-pre-trained backbone and visualise PCA feature maps.

    Creates a METERModel with the pre-trained backbone (no decoder needed
    for feature extraction — we still attach one so the forward() path is
    valid, but we only use the encoder output).
    """
    ckpt = CKPT_DIR / f"lejepa_{variant}_{dataset}.pth"
    assert ckpt.exists(), \
        f"Run pre-training first: lejepa_{variant}_{dataset}.pth not found"

    # Build a full model just to use its encoder
    model = METERModel(variant=variant, dataset=dataset).to(DEVICE)
    model.encoder.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()
    print(f"Loaded pre-trained backbone for PCA probing ({variant}/{dataset})")

    test_loader = get_loader(dataset, "test", "eval", batch_size=8)

    # Visualise skip features at different scales
    for skip_idx, scale in [(1, "H/4"), (2, "H/8"), (3, "H/16")]:
        fig = visualise_pca_probing(model, test_loader,
                                    n_images=4, skip_idx=skip_idx)
        fig.suptitle(
            f"LeJEPA zero-shot PCA — skip y{skip_idx} ({scale})  "
            f"[{variant}/{dataset}]",
            fontsize=13, y=1.01)
        plt.savefig(CKPT_DIR / f"pca_y{skip_idx}_{variant}_{dataset}.png",
                    bbox_inches="tight", dpi=120)
        plt.show()


# ── Run PCA probing for xxs on both datasets ──────────────────────────
for ds in DATASETS:
    run_pca_probing(variant="xxs", dataset=ds)


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Phase 5 — Full evaluation, comparison table, and convergence plots
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def load_finetuned(variant: str, dataset: str, tag: str) -> METERModel:
    """Load a fine-tuned checkpoint. tag: 'lejepa' or 'random'."""
    path  = CKPT_DIR / f"meter_{variant}_{dataset}_{tag}.pth"
    assert path.exists(), f"Checkpoint not found: {path}"
    model = METERModel(variant=variant, dataset=dataset).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model


def run_full_comparison(variants=None, datasets=None):
    """Evaluate all models and print a summary table.

    Models compared per (variant, dataset):
        A — Supervised METER (provided pre-trained weights)
        B — LeJEPA-METER     (SSL pre-train → fine-tune)
        C — Random-init METER (random → fine-tune, ablation)
    """
    if variants is None: variants = VARIANTS[:1]
    if datasets is None: datasets = DATASETS

    all_results = {}

    for variant in variants:
        for ds in datasets:
            max_d = NYUDataset.NYU_MAX_DEPTH if ds == "nyu" else KITTIDataset.KITTI_MAX_DEPTH

            for tag, loader_fn in [
                ("supervised", lambda: load_meter_baseline(variant, ds)),
                ("lejepa",     lambda: load_finetuned(variant, ds, "lejepa")),
                ("random",     lambda: load_finetuned(variant, ds, "random")),
            ]:
                try:
                    model = loader_fn()
                    m     = evaluate(model, ds)
                    key   = f"{tag}_{variant}_{ds}"
                    all_results[key] = m
                    print_metrics(f"{tag.upper():>12} METER-{variant.upper()} / {ds.upper()}", m)
                except AssertionError as e:
                    print(f"  Skipped {tag}/{variant}/{ds}: {e}")

    return all_results


# ── Print formatted comparison table ──────────────────────────────────
def print_comparison_table(results: dict):
    header = f"{'Model':<35} {'RMSE':>8} {'AbsRel':>8} {'δ₁(%)':>8}"
    sep    = "─" * len(header)
    print(f"\n{sep}\n{header}\n{sep}")
    for key, m in results.items():
        print(f"  {key:<33} {m['rmse']:>8.4f} {m['abs_rel']:>8.4f}"
              f" {m['delta1']*100:>8.2f}")
    print(sep + "\n")


# ── Plot convergence curves ────────────────────────────────────────────
def plot_convergence(histories: dict):
    """histories: {label: list_of_epoch_losses}"""
    fig, ax = plt.subplots(figsize=(9, 5))
    for label, losses in histories.items():
        ax.plot(range(1, len(losses) + 1), losses, label=label)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Fine-tuning loss")
    ax.set_title("Convergence: LeJEPA vs. random init")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(CKPT_DIR / "convergence.png", dpi=120)
    plt.show()


# ── Qualitative depth visualisation ───────────────────────────────────
@torch.no_grad()
def qualitative_comparison(models: dict, dataset: str, n: int = 4):
    """Side-by-side depth predictions for n test images.

    Args:
        models: {label: METERModel}
    """
    loader = get_loader(dataset, "test", "eval", batch_size=n)
    rgb, gt = next(iter(loader))
    rgb = rgb.to(DEVICE)

    n_models = len(models)
    fig, axes = plt.subplots(n, 2 + n_models,
                              figsize=(4 * (2 + n_models), 4 * n))
    for i in range(n):
        axes[i, 0].imshow(denorm(rgb[i]))
        axes[i, 0].set_title("RGB")
        axes[i, 1].imshow(gt[i, 0].cpu(), cmap="plasma")
        axes[i, 1].set_title("GT depth")
        for j, (label, model) in enumerate(models.items()):
            pred = model(rgb[i:i+1])
            pred = F.interpolate(pred, size=gt.shape[-2:],
                                 mode="bilinear", align_corners=False)
            axes[i, 2 + j].imshow(pred[0, 0].cpu(), cmap="plasma")
            axes[i, 2 + j].set_title(label)
        for ax in axes[i]:
            ax.axis("off")
    plt.suptitle(f"Qualitative comparison — {dataset.upper()}", y=1.01)
    plt.tight_layout()
    plt.savefig(CKPT_DIR / f"qualitative_{dataset}.png",
                bbox_inches="tight", dpi=120)
    plt.show()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Run full comparison (execute after all training is complete)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

all_results = run_full_comparison()
print_comparison_table(all_results)

# ── Convergence plots (requires histories saved during training) ───────
# Uncomment once fine-tuning histories are available:
# plot_convergence({
#     "LeJEPA-METER (xxs/nyu)": lejepa_history["loss"],
#     "Random-init  (xxs/nyu)": random_history["loss"],
# })

# ── Qualitative visualisation ──────────────────────────────────────────
for ds in DATASETS:
    try:
        models = {
            "Supervised": load_meter_baseline("xxs", ds),
            "LeJEPA":     load_finetuned("xxs", ds, "lejepa"),
            "Random":     load_finetuned("xxs", ds, "random"),
        }
        qualitative_comparison(models, ds)
    except AssertionError as e:
        print(f"Skipping qualitative ({ds}): {e}")
